In [5]:
import warnings
import tensorflow as tf, tensorflow_hub as hub
warnings.filterwarnings("ignore")
tf.get_logger().setLevel('ERROR')

print(tf.__version__)
model = hub.load("https://tfhub.dev/google/yamnet/1")
print("✅ loaded")


2.15.0


ValueError: Trying to load a model of incompatible/unknown type. 'C:\Users\takan\AppData\Local\Temp\tfhub_modules\9616fd04ec2360621642ef9455b84f4b668e219e' contains neither 'saved_model.pb' nor 'saved_model.pbtxt'.

In [ ]:
import sounddevice as sd
sd.query_devices()

In [ ]:
import sounddevice as sd
import tensorflow as tf
import tensorflow_hub as hub
import pandas as pd
import matplotlib.pyplot as plt

# モデルとラベルを関数内で完結させる
def stream_detect(chunk=3, duration=10.0):
    sr = 192000
    sd.default.device = (24, None)
    model = hub.load("https://tfhub.dev/google/yamnet/1")
    url = "https://raw.githubusercontent.com/robertanto/Real-Time-Sound-Event-Detection/main/keras_yamnet/yamnet_class_map.csv"
    labels = pd.read_csv(url)['display_name'].tolist()

    print("🎙️ Streaming detection start...")
    with sd.InputStream(samplerate=sr, channels=1, dtype='float32') as stream:
        for _ in range(int(duration / chunk)):
            audio, _ = stream.read(int(sr * chunk))
            scores, _, _ = model(audio[:, 0])
            mean_scores = tf.reduce_mean(scores, axis=0)
            top = int(tf.argmax(mean_scores))
            label = labels[top]
            prob = float(mean_scores[top])
            print(f"{label} ({prob:.2f})")
            plt.plot(audio)
            plt.show()
    top5 = tf.argsort(mean_scores, direction='DESCENDING')[:5]
    for i in top5:
        print(f"{labels[int(i)]}: {mean_scores[i].numpy():.2f}")
stream_detect(chunk=3, duration=10.0)



In [3]:
import sounddevice as sd
import tensorflow as tf
import tensorflow_hub as hub
import pandas as pd
import numpy as np
import scipy.signal
import time

def stream_detect(threshold=0.7):

    mic_sr = 192000
    yamnet_sr = 16000
    chunk = 0.96
    cooldown = 2.0

    sd.default.device = (24, None)

    model = hub.load("https://tfhub.dev/google/yamnet/1")

    url = "https://raw.githubusercontent.com/robertanto/Real-Time-Sound-Event-Detection/main/keras_yamnet/yamnet_class_map.csv"
    labels = pd.read_csv(url)['display_name'].tolist()

    last_print = {}

    print("Streaming detection start")

    with sd.InputStream(samplerate=mic_sr, channels=1, dtype='float32') as stream:

        while True:

            audio, _ = stream.read(int(mic_sr * chunk))
            audio = audio[:, 0]

            audio = scipy.signal.resample_poly(audio, yamnet_sr, mic_sr)

            scores, _, _ = model(audio)

            mean_scores = tf.reduce_mean(scores, axis=0)

            top = int(tf.argmax(mean_scores))
            prob = float(mean_scores[top])
            label = labels[top]

            now = time.time()

            if prob >= threshold:

                if label not in last_print or now - last_print[label] > cooldown:
                    print(f"{label} ({prob:.2f})")
                    last_print[label] = now


stream_detect()

Streaming detection start
Pant (0.80)


KeyboardInterrupt: 

In [2]:
!pip list

Package                   Version
------------------------- --------------
anyio                     4.4.0
archspec                  0.2.3
argon2-cffi               23.1.0
argon2-cffi-bindings      21.2.0
arrow                     1.3.0
asttokens                 2.4.1
async-lru                 2.0.4
attrs                     24.2.0
Babel                     2.14.0
beautifulsoup4            4.12.3
bleach                    6.1.0
boltons                   24.0.0
Brotli                    1.1.0
cached-property           1.5.2
certifi                   2024.7.4
cffi                      1.17.0
charset-normalizer        3.3.2
colorama                  0.4.6
comm                      0.2.2
conda                     24.7.1
conda-libmamba-solver     24.7.0
conda-package-handling    2.3.0
conda_package_streaming   0.10.0
contourpy                 1.2.1
cycler                    0.12.1
debugpy                   1.8.5
decorator                 5.1.1
defusedxml                0.7.1
distro         

In [3]:
import sys
print(sys.executable)

C:\GitHub\embodied-YAMNet\yamnet-py311\Scripts\python.exe


In [4]:
import sys
!{sys.executable} -m pip list

Package                      Version
---------------------------- -----------
absl-py                      2.4.0
asttokens                    3.0.1
astunparse                   1.6.3
certifi                      2026.6.17
cffi                         2.0.0
charset-normalizer           3.4.7
colorama                     0.4.6
comm                         0.2.3
contourpy                    1.3.3
cryptography                 49.0.0
cycler                       0.12.1
debugpy                      1.8.21
decorator                    5.3.1
executing                    2.2.1
flatbuffers                  25.12.19
fonttools                    4.63.0
gast                         0.7.0
google-auth                  2.55.0
google-auth-oauthlib         1.4.0
google-pasta                 0.2.0
grpcio                       1.81.1
h5py                         3.16.0
idna                         3.18
ipykernel                    7.3.0
ipython                      9.14.1
ipython_pygments_lexers      1.1.